# AWS access 

# AWS + Jupyter Practice Guide

A reference for connecting to AWS from Jupyter and practicing S3 operations.

---

## 1. Authentication Setup (One-Time Per Session)

Before running this notebook, authenticate via the terminal:

```bash
aws login
```

This opens a browser, logs you in, and stores temporary credentials in `~/.aws/`. Sessions typically expire after a few hours — just re-run `aws login` and restart the kernel when needed.

**Required dependencies** (run once per environment):

```python
import sys
!{sys.executable} -m pip install boto3 "botocore[crt]" pandas
```

---

## 2. Verify Your Connection

Confirm boto3 can see your credentials and identify your account.

```python
import boto3

sts = boto3.client("sts")
identity = sts.get_caller_identity()

print(f"Account: {identity['Account']}")
print(f"User ARN: {identity['Arn']}")
print(f"Region:  {boto3.Session().region_name}")
```

If this returns your account info, you're fully connected. ✅

---

## 3. Create an S3 Client

```python
import boto3

s3 = boto3.client("s3")
```

This client uses your `aws login` credentials automatically — no keys needed.

---

## 4. Create a Bucket (Programmatically)

Bucket names must be **globally unique across all of AWS**. Use a timestamp or distinctive suffix.

```python
from datetime import datetime
from botocore.exceptions import ClientError

def create_bucket(bucket_name, region=None):
    """Create an S3 bucket, handling region and common errors."""
    try:
        if region is None or region == "us-east-1":
            client = boto3.client("s3")
            client.create_bucket(Bucket=bucket_name)
        else:
            client = boto3.client("s3", region_name=region)
            client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"✅ Created bucket: {bucket_name}")
        return True

    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code == "BucketAlreadyOwnedByYou":
            print(f"⚠️  You already own '{bucket_name}'")
        elif code == "BucketAlreadyExists":
            print(f"❌ '{bucket_name}' is taken — try a different name")
        else:
            print(f"❌ Error: {e}")
        return False

# Create a uniquely named practice bucket
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
bucket = f"courtney-practice-{timestamp}"

create_bucket(bucket, region="us-east-1")
```

**Bucket naming rules:**
- 3–63 characters
- Lowercase letters, numbers, hyphens only
- Must start and end with a letter or number
- No underscores, no uppercase

---

## 5. Full S3 Lifecycle (CRUD)

Walk through the complete create-read-update-delete cycle.

### Step 1 — List All Buckets

```python
for b in s3.list_buckets()["Buckets"]:
    print(b["Name"])
```

### Step 2 — Upload an Object

```python
s3.put_object(
    Bucket=bucket,
    Key="hello.txt",
    Body="Hello from Jupyter!"
)
print("Uploaded hello.txt")
```

### Step 3 — Read the Object Back

```python
obj = s3.get_object(Bucket=bucket, Key="hello.txt")
content = obj["Body"].read().decode("utf-8")
print(content)
```

### Step 4 — Upload a Local File

```python
# Replace 'local_file.csv' with an actual file on your laptop
s3.upload_file("local_file.csv", bucket, "data/local_file.csv")
print("Uploaded local file to s3://" + bucket + "/data/local_file.csv")
```

### Step 5 — Read a CSV into pandas

```python
import pandas as pd
from io import StringIO

obj = s3.get_object(Bucket=bucket, Key="data/local_file.csv")
df = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))

df.head()
```

### Step 6 — Transform and Write Back

```python
# Example: filter rows and save as a new CSV in S3
df_filtered = df[df["some_column"] > 100]

csv_buffer = StringIO()
df_filtered.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=bucket,
    Key="data/filtered.csv",
    Body=csv_buffer.getvalue()
)
print("Wrote filtered.csv back to S3")
```

### Step 7 — List Objects in the Bucket

```python
response = s3.list_objects_v2(Bucket=bucket)
for obj in response.get("Contents", []):
    print(f"{obj['Key']:40} {obj['Size']:>10} bytes")
```

### Step 8 — Download an Object Locally

```python
s3.download_file(bucket, "data/filtered.csv", "filtered_local.csv")
print("Downloaded to ./filtered_local.csv")
```

### Step 9 — Delete Individual Objects

```python
s3.delete_object(Bucket=bucket, Key="hello.txt")
s3.delete_object(Bucket=bucket, Key="data/local_file.csv")
s3.delete_object(Bucket=bucket, Key="data/filtered.csv")
print("Objects deleted")
```

### Step 10 — Delete the Bucket

A bucket must be **empty** before it can be deleted.

```python
s3.delete_bucket(Bucket=bucket)
print(f"Deleted bucket: {bucket}")
```

---

## 6. Troubleshooting Quick Reference

| Error | Fix |
|---|---|
| `ModuleNotFoundError: No module named 'boto3'` | `!{sys.executable} -m pip install boto3` |
| `MissingDependencyException ... botocore[crt]` | `!{sys.executable} -m pip install "botocore[crt]"` |
| `NoCredentialsError` | Run `aws login` in terminal, restart kernel |
| `SSOTokenLoadError` / token expired | Run `aws login` again |
| `BucketAlreadyExists` | Pick a more unique bucket name |
| `IllegalLocationConstraintException` | Region in code doesn't match your default — check `boto3.Session().region_name` |

---

## 7. Session Reminders

- 🔄 `aws login` credentials are **temporary** — re-authenticate every few hours
- 🔁 After re-authenticating, **restart your Jupyter kernel**
- 🧹 Always clean up practice buckets and objects to avoid clutter and unexpected charges
- 🔒 Never commit credentials or keys to git — `aws login` keeps you safe by design

In [1]:
import sys
print("Python:", sys.executable)

import boto3
print("boto3 version:", boto3.__version__)

Python: /usr/local/bin/python
boto3 version: 1.43.18


In [2]:
import sys
!{sys.executable} -m pip install pandas python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import boto3

sts = boto3.client("sts")
identity = sts.get_caller_identity()

print(f"Account: {identity['Account']}")
print(f"User ARN: {identity['Arn']}")
print(f"Region:  {boto3.Session().region_name}")

Account: 710472718354
User ARN: arn:aws:iam::710472718354:root
Region:  us-east-1


In [4]:
import sys
!{sys.executable} -m pip install "botocore[crt]"

Defaulting to user installation because normal site-packages is not writeable


In [5]:
s3 = boto3.client("s3")
for b in s3.list_buckets()["Buckets"]:
    print(b["Name"])

courtney-practice-bucket-710472718354-us-east-2-an


In [ ]:
s3.upload_file("/Users/courtneysmith/Documents/Projects/DataPractice/data/raw/insurance_policies.csv","courtney-practice-bucket-710472718354-us-east-2-an","data/raw/insurance_policies.csv")

In [7]:
s3.upload_file("/Users/courtneysmith/Documents/Projects/DataPractice/data/raw/fake_data.csv","courtney-practice-bucket-710472718354-us-east-2-an","data/raw/fake_data.csv")

In [ ]:
# List objects in a S3 Bucket
s3 = boto3.client("s3")
for obj in s3.list_objects_v2(Bucket=bucket).get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

data/raw/fake_data.csv 12438 bytes
data/raw/insurance_policies.csv 1672978 bytes


In [11]:
insurance_data = 'data/raw/insurance_policies.csv'
fake_data_raw = 'data/raw/fake_data.csv'

In [9]:
bucket = 'courtney-practice-bucket-710472718354-us-east-2-an'

In [ ]:

import pandas as pd 
from io import StringIO

obj = s3.get_object(Bucket="courtney-practice-bucket-710472718354-us-east-2-an", Key="data/raw/insurance_policies.csv")
df = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))
df.head()

,first_name,last_name,date_of_birth,age,gender,marital_status,education,occupation,annual_income,credit_score,...,acquisition_channel,agent_id,has_multi_policy_discount,num_claims,total_claims_paid,last_claim_date,primary_claim_type,has_safe_driver_discount,nps_score,customer_lifetime_value
0,Joshua,Walker,1941-12-14,84,F,Single,Bachelor,"Civil engineer, consulting",131900.0,781,...,Phone,AGT-1114,True,0,0.0,NaN,NaN,False,2,73517.99
1,Teresa,Gray,1974-09-02,51,M,Single,Doctorate,Sub,42100.0,548,...,Online,AGT-1016,False,0,0.0,NaN,NaN,True,0,17627.76
2,Nancy,Edwards,2000-09-12,25,M,Single,Some College,Restaurant manager,69300.0,669,...,Agent,AGT-1258,True,0,0.0,NaN,NaN,False,0,5652.89
3,Joseph,Davidson,1974-09-20,51,F,Married,Doctorate,Clinical research associate,86600.0,733,...,Agent,AGT-1112,True,0,0.0,NaN,NaN,False,9,15312.46
4,Sarah,Campos,2002-01-26,24,M,Divorced,High School,"Engineer, control and instrumentation",100300.0,794,...,Agent,AGT-1357,True,0,0.0,NaN,NaN,False,5,23403.52


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 39 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   first_name                 5000 non-null   str    
 1   last_name                  5000 non-null   str    
 2   date_of_birth              5000 non-null   str    
 3   age                        5000 non-null   int64  
 4   gender                     5000 non-null   str    
 5   marital_status             5000 non-null   str    
 6   education                  5000 non-null   str    
 7   occupation                 5000 non-null   str    
 8   annual_income              5000 non-null   float64
 9   credit_score               5000 non-null   int64  
 10  email                      5000 non-null   str    
 11  phone                      5000 non-null   str    
 12  address                    5000 non-null   str    
 13  city                       5000 non-null   str    
 14  sta

In [ ]:
df_filtered = df[df["credit_score"] >= 718]

csv_buffer = StringIO()
df_filtered.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket="courtney-practice-bucket-710472718354-us-east-2-an",
    Key="data/raw/insurance_policies_filtered.csv",
    Body=csv_buffer.getvalue()
)

{'ResponseMetadata': {'RequestId': 'JAFEH7W4Z6H109S3',
  'HostId': '1jg5kQeBKrf9Xn4K8KttrsslvUmc6/Id48ohllAC69XxO5EhyaXJFQRET4O7XxCKRFPPEZ7MwYY=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '1jg5kQeBKrf9Xn4K8KttrsslvUmc6/Id48ohllAC69XxO5EhyaXJFQRET4O7XxCKRFPPEZ7MwYY=',
   'x-amz-request-id': 'JAFEH7W4Z6H109S3',
   'date': 'Sun, 31 May 2026 02:13:48 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"b70e11eee4bce15c8e2dd56151b5ced7"',
   'x-amz-checksum-crc32': 'ZYreOw==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"b70e11eee4bce15c8e2dd56151b5ced7"',
 'ChecksumCRC32': 'ZYreOw==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [ ]:
# List objects in the bucket
bucket = "courtney-practice-bucket-710472718354-us-east-2-an"

for obj in s3.list_objects_v2(Bucket=bucket).get("Contents", []):
    print(obj["Key"])

data/raw/insurance_policies.csv
data/raw/insurance_policies_filtered.csv


In [ ]:
obj = s3.get_object(Bucket="courtney-practice-bucket-710472718354-us-east-2-an", Key="data/raw/insurance_policies_filtered.csv")
df = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))
df.head()

,first_name,last_name,date_of_birth,age,gender,marital_status,education,occupation,annual_income,credit_score,...,acquisition_channel,agent_id,has_multi_policy_discount,num_claims,total_claims_paid,last_claim_date,primary_claim_type,has_safe_driver_discount,nps_score,customer_lifetime_value
0,Joshua,Walker,1941-12-14,84,F,Single,Bachelor,"Civil engineer, consulting",131900.0,781,...,Phone,AGT-1114,True,0,0.0,NaN,NaN,False,2,73517.99
1,Joseph,Davidson,1974-09-20,51,F,Married,Doctorate,Clinical research associate,86600.0,733,...,Agent,AGT-1112,True,0,0.0,NaN,NaN,False,9,15312.46
2,Sarah,Campos,2002-01-26,24,M,Divorced,High School,"Engineer, control and instrumentation",100300.0,794,...,Agent,AGT-1357,True,0,0.0,NaN,NaN,False,5,23403.52
3,Nicholas,Galloway,1944-02-08,82,M,Divorced,Bachelor,Geochemist,40400.0,779,...,Agent,AGT-1135,False,0,0.0,NaN,NaN,True,0,79639.52
4,Shelly,Hudson,1966-03-12,60,F,Single,Master,Technical sales engineer,51300.0,773,...,Agent,AGT-1282,False,0,0.0,NaN,NaN,False,4,54774.30


In [ ]:
# Delete the object
s3.delete_object(Bucket=bucket, Key="data/raw/insurance_policies_filtered.csv")

{'ResponseMetadata': {'RequestId': 'JAF11ZKHEC6D5T6J',
  'HostId': 'LX0tYP0h5f7m8DPgOTF6RhiRwWZY4lEhDKr3LOCrEVG6UInvADjmDU/42DYRqG6O13E4HrJKCYQ=',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'LX0tYP0h5f7m8DPgOTF6RhiRwWZY4lEhDKr3LOCrEVG6UInvADjmDU/42DYRqG6O13E4HrJKCYQ=',
   'x-amz-request-id': 'JAF11ZKHEC6D5T6J',
   'date': 'Sun, 31 May 2026 02:13:48 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}

In [ ]:
s3.delete_object(Bucket=bucket, Key="data/raw/insurance_policies.csv")

{'ResponseMetadata': {'RequestId': 'JAF5ZPW884S3R0JV',
  'HostId': 'MGKvvEtqm8JrmxWNuchYp3ZlMVI7n6lYzdTk6GlRhlKsVxL4s1IMCbewLRHH1lBq75dM/Me9pf0=',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'MGKvvEtqm8JrmxWNuchYp3ZlMVI7n6lYzdTk6GlRhlKsVxL4s1IMCbewLRHH1lBq75dM/Me9pf0=',
   'x-amz-request-id': 'JAF5ZPW884S3R0JV',
   'date': 'Sun, 31 May 2026 02:13:48 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}

In [ ]:
for obj in s3.list_objects_v2(Bucket=bucket).get("Contents", []):
    print(obj["Key"])